Notes:



1. Lets call this Ontology registry. Treat OWL ontologies and SHACL shapes as equvialent ontology files.
2. where are these files located in the project.  we should put them all in the same place.
3. The how to run the notebook section needs to be the same as all the other notebooks.  not sure why you keep reverting to this very old intro.
4. the librarties should be described like skos_owl, or shos_shacl.  dont treat them like pairs.
5. access everything from starlayergraph.ontology.  e.g. starlayergraph.ontology.list_ontologies() no need for an import.  
6. we dont wnat the object (e.g. skos) to validate.  we should use shacl validate witht he shape file and the data file or use inference with the ontology file.  we should show both below.

# The SHACL Library Registry — User Guide

`starlayergraph` ships several ontology/SHACL-shapes pairs - `manch:` (Manchester Syntax's own AST as RDF), `skos:` (the W3C SKOS Reference's formal axioms and integrity conditions), `salg:` (a SPARQL query's own algebra as RDF, from the sibling `starsparql` package), and the SHACL 1.2 meta-shapes themselves (from `starshacl`) - each with its own loader functions scattered across three packages. The registry (`starlayergraph.ontology.list_shacl_libraries()`/`get_shacl_library()`) is a single catalog over all of them, so a caller can discover and load any one by name without knowing which package or file it actually lives in.

See the [Manchester Syntax as RDF guide](05g-manchester-syntax-as-rdf.ipynb) for what one specific library (`manch:`) actually looks like in depth - this guide is about the catalog over all of them, not any one library's own design.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. This guide's validation steps need `pyshacl` (`pip install -e packages/graph[test]` from a local checkout).
3. Run cells from top to bottom — later steps reuse variables from earlier ones.

## Step 1 — list what's registered

In [1]:
from starlayergraph.ontology import list_shacl_libraries

list_shacl_libraries()

('manchester', 'shacl', 'skos', 'sparql')

## Step 2 — look up one library

`get_shacl_library(name)` returns a `ShaclLibrary` - a name, a description, its own vocabulary namespace, and lazy `ontology_graph()`/`shapes_graph()`/`validate()` loaders. "Lazy" matters here: listing libraries never needs `pyshacl` installed at all - only actually calling `shapes_graph()`/`validate()` does, the same friendly, actionable error each library's own module already raises if it's missing.

In [2]:
from starlayergraph.ontology import get_shacl_library

skos = get_shacl_library('skos')
print(skos.name)
print(skos.description)
print(skos.namespace)

skos
The W3C SKOS Reference's own formal axioms and numbered Documented Consistency and Integrity Conditions, over the real skos: namespace.
http://www.w3.org/2004/02/skos/core#


## Step 3 — load a library's ontology and shapes

Both come back as fresh, plain `rdflib.Graph`s - real, standalone artifacts usable by any RDF tool, not something the registry itself invents.

In [3]:
skos_ontology = skos.ontology_graph()
skos_shapes = skos.shapes_graph()
print('ontology triples:', len(skos_ontology))
print('shapes triples:  ', len(skos_shapes))

ontology triples: 90
shapes triples:   79


## Step 4 — validate real data against a registered library

A tiny, genuinely valid SKOS thesaurus - a `ConceptScheme` with one `Concept`, in scope, with a labeled preferred term.

In [4]:
from starlayergraph import StarLayerGraph, Namespace
from rdflib import RDF, Literal
from rdflib.namespace import SKOS

EX = Namespace('http://example.org/animals#')

g = StarLayerGraph()
g.bind('ex', EX)
g.add((EX.scheme, RDF.type, SKOS.ConceptScheme))
g.add((EX.cat, RDF.type, SKOS.Concept))
g.add((EX.cat, SKOS.inScheme, EX.scheme))
g.add((EX.cat, SKOS.prefLabel, Literal('Cat', lang='en')))

conforms, results_graph, results_text = skos.validate(g)
print('conforms:', conforms)

conforms: True


## Step 5 — the `"shacl"` library validates *shapes graphs*, not data

Every other registered library's `validate()` checks a data graph against that library's own shapes. `"shacl"` is the one entry with a genuinely different meaning: it has no separate OWL ontology of its own (`ontology_graph` is `None`, not a stub), and its `validate()` checks whether a *shapes graph* - any shapes graph, including one from another registered library - is itself well-formed SHACL (SHACL's own meta-shapes, "meta-validation"). Here it checks the SKOS shapes graph loaded above.

In [5]:
shacl_lib = get_shacl_library('shacl')
print(shacl_lib.description)
print('has its own ontology:', shacl_lib.ontology_graph is not None)

meta_conforms, _meta_report_graph, _meta_report_text = shacl_lib.validate(skos_shapes)
print('skos_shapes is well-formed SHACL:', meta_conforms)

The SHACL 1.2 meta-shapes themselves (starshacl's own shapes-about-shapes, validating a shapes graph's own well-formedness) - no separate OWL ontology exists for SHACL's own vocabulary anywhere in this stack, so ontology_graph is None for this one entry, not a stub.
has its own ontology: False
skos_shapes is well-formed SHACL: True


## Step 6 — an unknown library name fails loudly

`get_shacl_library()` raises `KeyError` naming every valid choice, rather than returning `None` - a typo shouldn't surface as a confusing `AttributeError` two lines later.

In [6]:
try:
    get_shacl_library('turtle')
except KeyError as e:
    print(f'Rejected as expected: {e}')

Rejected as expected: "'turtle' is not a registered SHACL library - choices are ('manchester', 'shacl', 'skos', 'sparql')"


## Further work

- **`StarLayerGraph.derive()`** is the complementary tool this registry doesn't cover yet: given a graph's own data (not a hand-authored vocabulary), it infers a SHACL shape directly - see its own docstring and `tests/unit/test_shape_derivation.py`.
- Not registered yet, deliberately: a `"generic"` RDF-editor shape (doesn't exist as a real artifact yet) and an `"owl"` entry (names nothing concrete in this codebase) - see `packages/graph/starlayergraph/ontology/registry.py`'s own docstring.